# Baseline vs small hyperparameter sweep

This notebook runs a deterministic, small logistic-regression task using the
sample ML helpers. It compares one baseline configuration against a compact
grid sweep and writes all generated artifacts under
`artifacts/agent_rounds/20260615_round2/ml_sweep/`.

The evidence to inspect is:

- `dataset_profile.json`
- `baseline_metrics.json`
- `sweep_results.json`
- `sweep_results.csv`
- `comparison_summary.json`


In [1]:
from pathlib import Path

from samples.ml_tasks.common import (
    FEATURES,
    display_html,
    generate_classification_rows,
    save_json,
    split_rows,
)

ARTIFACT_DIR = Path("artifacts/agent_rounds/20260615_round2/ml_sweep")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

ROW_COUNT = 720
DATA_SEED = 20260615
SPLIT_SEED = 41

rows = generate_classification_rows(n=ROW_COUNT, seed=DATA_SEED)
train_rows, valid_rows = split_rows(rows, train_frac=0.72, seed=SPLIT_SEED)

profile = {
    "row_count": len(rows),
    "train_rows": len(train_rows),
    "valid_rows": len(valid_rows),
    "positive_rate_all": sum(row["label"] for row in rows) / len(rows),
    "positive_rate_train": sum(row["label"] for row in train_rows) / len(train_rows),
    "positive_rate_valid": sum(row["label"] for row in valid_rows) / len(valid_rows),
    "features": FEATURES,
    "data_seed": DATA_SEED,
    "split_seed": SPLIT_SEED,
}

profile_path = save_json(ARTIFACT_DIR / "dataset_profile.json", profile)

print(f"artifact_dir: {ARTIFACT_DIR}")
print(f"profile_artifact: {profile_path}")
print(
    "rows: "
    f"train={profile['train_rows']} valid={profile['valid_rows']} "
    f"valid_positive_rate={profile['positive_rate_valid']:.3f}"
)

display_html(
    "<h3>Dataset profile</h3>"
    "<table>"
    f"<tr><td>rows</td><td>{profile['row_count']}</td></tr>"
    f"<tr><td>train rows</td><td>{profile['train_rows']}</td></tr>"
    f"<tr><td>validation rows</td><td>{profile['valid_rows']}</td></tr>"
    f"<tr><td>validation positive rate</td><td>{profile['positive_rate_valid']:.3f}</td></tr>"
    "</table>"
)


artifact_dir: artifacts/agent_rounds/20260615_round2/ml_sweep
profile_artifact: artifacts/agent_rounds/20260615_round2/ml_sweep/dataset_profile.json
rows: train=518 valid=202 valid_positive_rate=0.490


rows,720
train rows,518
validation rows,202
validation positive rate,0.490


In [2]:
from samples.ml_tasks.common import (
    classification_metrics,
    metrics_table,
    save_json,
    score_rows,
    train_logistic,
)

baseline_config = {"lr": 0.04, "l2": 0.001, "epochs": 100}
baseline_model = train_logistic(train_rows, **baseline_config)
baseline_scores = score_rows(valid_rows, baseline_model)
baseline_metrics = classification_metrics(baseline_scores)

baseline_record = {
    "config": baseline_config,
    "metrics": baseline_metrics,
    "model": {
        "features": baseline_model["features"],
        "weights": baseline_model["weights"],
        "bias": baseline_model["bias"],
        "history": baseline_model["history"],
    },
}

baseline_path = save_json(ARTIFACT_DIR / "baseline_metrics.json", baseline_record)

print(f"baseline_artifact: {baseline_path}")
print(
    "baseline: "
    f"f1={baseline_metrics['f1']:.4f} "
    f"accuracy={baseline_metrics['accuracy']:.4f} "
    f"brier={baseline_metrics['brier']:.4f}"
)

metrics_table("Baseline validation metrics", baseline_metrics)


baseline_artifact: artifacts/agent_rounds/20260615_round2/ml_sweep/baseline_metrics.json
baseline: f1=0.7640 accuracy=0.7921 brier=0.1673


tp,68
fp,11
tn,92
fn,31
precision,0.8608
recall,0.6869
accuracy,0.7921
f1,0.7640
brier,0.1673


In [3]:
import itertools

from samples.ml_tasks.common import (
    classification_metrics,
    display_html,
    save_json,
    score_rows,
    train_logistic,
    write_csv,
)

learning_rates = [0.02, 0.04, 0.08]
l2_values = [0.0, 0.001, 0.01]
epoch_values = [80, 140]

sweep_results = []
for lr, l2, epochs in itertools.product(learning_rates, l2_values, epoch_values):
    model = train_logistic(train_rows, lr=lr, l2=l2, epochs=epochs)
    metrics = classification_metrics(score_rows(valid_rows, model))
    row = {"lr": lr, "l2": l2, "epochs": epochs, **metrics}
    sweep_results.append(row)
    print(
        f"lr={lr:.3f} l2={l2:.3f} epochs={epochs} "
        f"f1={metrics['f1']:.4f} acc={metrics['accuracy']:.4f} "
        f"brier={metrics['brier']:.4f}"
    )

best_config = max(
    sweep_results,
    key=lambda row: (row["f1"], row["accuracy"], -row["brier"], -row["l2"]),
)
ranked_results = sorted(
    sweep_results,
    key=lambda row: (row["f1"], row["accuracy"], -row["brier"]),
    reverse=True,
)

sweep_path = save_json(
    ARTIFACT_DIR / "sweep_results.json",
    {
        "grid": {
            "learning_rates": learning_rates,
            "l2_values": l2_values,
            "epoch_values": epoch_values,
        },
        "results": sweep_results,
        "best": best_config,
    },
)
csv_path = write_csv(
    ARTIFACT_DIR / "sweep_results.csv",
    ranked_results,
    [
        "lr",
        "l2",
        "epochs",
        "tp",
        "fp",
        "tn",
        "fn",
        "precision",
        "recall",
        "accuracy",
        "f1",
        "brier",
    ],
)

comparison_summary = {
    "baseline": {"config": baseline_config, "metrics": baseline_metrics},
    "best_sweep": best_config,
    "delta": {
        "f1": best_config["f1"] - baseline_metrics["f1"],
        "accuracy": best_config["accuracy"] - baseline_metrics["accuracy"],
        "brier": best_config["brier"] - baseline_metrics["brier"],
    },
    "selected_by": "max f1, then accuracy, then lower brier",
}
summary_path = save_json(ARTIFACT_DIR / "comparison_summary.json", comparison_summary)

print(f"sweep_artifact: {sweep_path}")
print(f"sweep_csv: {csv_path}")
print(f"comparison_artifact: {summary_path}")
print(
    "best_sweep: "
    f"lr={best_config['lr']:.3f} l2={best_config['l2']:.3f} "
    f"epochs={best_config['epochs']} f1={best_config['f1']:.4f} "
    f"accuracy={best_config['accuracy']:.4f} brier={best_config['brier']:.4f}"
)
print(
    "delta_vs_baseline: "
    f"f1={comparison_summary['delta']['f1']:+.4f} "
    f"accuracy={comparison_summary['delta']['accuracy']:+.4f} "
    f"brier={comparison_summary['delta']['brier']:+.4f}"
)

rows_html = "\n".join(
    "<tr>"
    f"<td>{row['lr']}</td>"
    f"<td>{row['l2']}</td>"
    f"<td>{row['epochs']}</td>"
    f"<td>{row['f1']:.3f}</td>"
    f"<td>{row['accuracy']:.3f}</td>"
    f"<td>{row['brier']:.3f}</td>"
    "</tr>"
    for row in ranked_results[:8]
)
display_html(
    "<h3>Top sweep configurations</h3>"
    "<table>"
    "<tr><th>lr</th><th>l2</th><th>epochs</th><th>f1</th><th>accuracy</th><th>brier</th></tr>"
    f"{rows_html}"
    "</table>"
)


lr=0.020 l2=0.000 epochs=80 f1=0.7598 acc=0.7871 brier=0.1966
lr=0.020 l2=0.000 epochs=140 f1=0.7640 acc=0.7921 brier=0.1779
lr=0.020 l2=0.001 epochs=80 f1=0.7598 acc=0.7871 brier=0.1966
lr=0.020 l2=0.001 epochs=140 f1=0.7640 acc=0.7921 brier=0.1780
lr=0.020 l2=0.010 epochs=80 f1=0.7640 acc=0.7921 brier=0.1969
lr=0.020 l2=0.010 epochs=140 f1=0.7640 acc=0.7921 brier=0.1785
lr=0.040 l2=0.000 epochs=80 f1=0.7640 acc=0.7921 brier=0.1737
lr=0.040 l2=0.000 epochs=140 f1=0.7640 acc=0.7921 brier=0.1592
lr=0.040 l2=0.001 epochs=80 f1=0.7640 acc=0.7921 brier=0.1737
lr=0.040 l2=0.001 epochs=140 f1=0.7640 acc=0.7921 brier=0.1593
lr=0.040 l2=0.010 epochs=80 f1=0.7640 acc=0.7921 brier=0.1743
lr=0.040 l2=0.010 epochs=140 f1=0.7640 acc=0.7921 brier=0.1602
lr=0.080 l2=0.000 epochs=80 f1=0.7640 acc=0.7921 brier=0.1565
lr=0.080 l2=0.000 epochs=140 f1=0.7640 acc=0.7921 brier=0.1489
lr=0.080 l2=0.001 epochs=80 f1=0.7640 acc=0.7921 brier=0.1566
lr=0.080 l2=0.001 epochs=140 f1=0.7640 acc=0.7921 brier=0.1491


lr,l2,epochs,f1,accuracy,brier
0.08,0.0,140,0.764,0.792,0.149
0.08,0.001,140,0.764,0.792,0.149
0.08,0.01,140,0.764,0.792,0.150
0.08,0.0,80,0.764,0.792,0.156
0.08,0.001,80,0.764,0.792,0.157
0.08,0.01,80,0.764,0.792,0.158
0.04,0.0,140,0.764,0.792,0.159
0.04,0.001,140,0.764,0.792,0.159


## Findings

The validation split has 202 rows with a 0.490 positive rate. The baseline
configuration (`lr=0.04`, `l2=0.001`, `epochs=100`) reached F1 0.7640,
accuracy 0.7921, and Brier 0.1673.

The best sweep configuration by F1, accuracy, and lower Brier was `lr=0.08`,
`l2=0.0`, `epochs=140`. It matched the baseline F1 and accuracy but improved
Brier to 0.1489, a -0.0184 delta.

The sweep did not find a classification-threshold improvement on this small
grid, but it did find a better-calibrated model. I would select the swept
configuration if probability quality matters; otherwise the baseline is
equivalent on the threshold metrics measured here.
